# Occupancy MLP experiment — explained source copy

This notebook is a **standalone teaching copy** of `occupancy_experiment.py` from `model/src/`. It is outside the project repository and is not imported, executed, or used by the React game. The source code in the code cells is copied from the original file; the Markdown cells explain what each block does.

> Teaching reference only. Keep implementation and training in the original `.py` file. Do not use this notebook as an application dependency. The command-line entry-point cell is included for completeness, but the original script is the supported way to run the experiment.


## `<setup>`

This cell records the experiment purpose, imports the Python and NumPy utilities, and defines the board, fleet, snapshot schedule, and feature names. The script is an offline experiment: it does not select actions in React.


In [ ]:
"""Offline, NumPy-only ship-occupancy experiment for Adaptive AI Battleship.

This module deliberately does not choose shots or connect to the React game.
Training labels may inspect a generated hidden fleet; feature construction may
inspect only the point-in-time public shot history and remaining fleet sizes.
"""


## `<setup>`

This cell records the experiment purpose, imports the Python and NumPy utilities, and defines the board, fleet, snapshot schedule, and feature names. The script is an offline experiment: it does not select actions in React.


In [ ]:

from __future__ import annotations


## `<setup>`

This cell records the experiment purpose, imports the Python and NumPy utilities, and defines the board, fleet, snapshot schedule, and feature names. The script is an offline experiment: it does not select actions in React.


In [ ]:

import argparse


## `<setup>`

This cell records the experiment purpose, imports the Python and NumPy utilities, and defines the board, fleet, snapshot schedule, and feature names. The script is an offline experiment: it does not select actions in React.


In [ ]:
import json


## `<setup>`

This cell records the experiment purpose, imports the Python and NumPy utilities, and defines the board, fleet, snapshot schedule, and feature names. The script is an offline experiment: it does not select actions in React.


In [ ]:
from pathlib import Path


## `<setup>`

This cell records the experiment purpose, imports the Python and NumPy utilities, and defines the board, fleet, snapshot schedule, and feature names. The script is an offline experiment: it does not select actions in React.


In [ ]:
from time import perf_counter


## `<setup>`

This cell records the experiment purpose, imports the Python and NumPy utilities, and defines the board, fleet, snapshot schedule, and feature names. The script is an offline experiment: it does not select actions in React.


In [ ]:

import numpy as np


## `<setup>`

This cell records the experiment purpose, imports the Python and NumPy utilities, and defines the board, fleet, snapshot schedule, and feature names. The script is an offline experiment: it does not select actions in React.


In [ ]:


BOARD_SIZE = 10


## `<setup>`

This cell records the experiment purpose, imports the Python and NumPy utilities, and defines the board, fleet, snapshot schedule, and feature names. The script is an offline experiment: it does not select actions in React.


In [ ]:
FLEET = (5, 4, 3, 3, 2)


## `<setup>`

This cell records the experiment purpose, imports the Python and NumPy utilities, and defines the board, fleet, snapshot schedule, and feature names. The script is an offline experiment: it does not select actions in React.


In [ ]:
SNAPSHOTS = (0, 3, 6, 10, 14, 18, 23, 28, 34, 40)


## `<setup>`

This cell records the experiment purpose, imports the Python and NumPy utilities, and defines the board, fleet, snapshot schedule, and feature names. The script is an offline experiment: it does not select actions in React.


In [ ]:
FEATURE_NAMES = (
    "row_norm", "col_norm", "center_distance", "edge_distance", "is_corner",
    "row_parity", "col_parity", "shots_fraction", "observed_hits_fraction",
    "observed_misses_fraction", "adjacent_unsunk_hits", "adjacent_misses",
    "diagonal_unsunk_hits", "same_row_unsunk_hits", "same_col_unsunk_hits",
    "nearest_unsunk_hit_distance", "remaining_segments_fraction",
    "remaining_ships_fraction",
)


## `generate_fleet`

Generates a legal fleet with no overlapping ship cells. The optional placement style changes the distribution used to choose layouts; it does not become an AI input feature. These styles let the experiment evaluate different synthetic layouts.


In [ ]:


def generate_fleet(rng: np.random.Generator, style: str = "random") -> list[list[tuple[int, int]]]:
    """Generate a legal fleet. Style affects placement only, never features."""
    for _ in range(100):
        occupied: set[tuple[int, int]] = set()
        ships: list[list[tuple[int, int]]] = []
        valid = True
        for length in FLEET:
            placements = []
            for horizontal in (True, False):
                max_row = BOARD_SIZE if horizontal else BOARD_SIZE - length + 1
                max_col = BOARD_SIZE - length + 1 if horizontal else BOARD_SIZE
                for row in range(max_row):
                    for col in range(max_col):
                        cells = ([(row, col + i) for i in range(length)] if horizontal
                                 else [(row + i, col) for i in range(length)])
                        if not occupied.intersection(cells):
                            placements.append(cells)
            if not placements:
                valid = False
                break
            if style == "random":
                probabilities = None
            else:
                def preference(cells):
                    row = np.mean([r for r, _ in cells]) / 9
                    col = np.mean([c for _, c in cells]) / 9
                    edge = min(row, col, 1 - row, 1 - col)
                    if style == "edge": return np.exp(-8 * edge)
                    if style == "center": return np.exp(-8 * (abs(row - .5) + abs(col - .5)))
                    if style == "corner" : return np.exp(-5 * (row + col))
                    raise ValueError(f"Unknown fleet style: {style}")
                probabilities = np.asarray([preference(cells) for cells in placements], dtype=float)
                probabilities /= probabilities.sum()
            ship = placements[int(rng.choice(len(placements), p=probabilities))]
            ships.append(ship)
            occupied.update(ship)
        if valid:
            return ships
    raise RuntimeError("Could not generate a legal fleet after 100 layout attempts")


## `neighbors`

Returns the in-bounds orthogonal neighbors of a cell, or includes diagonals when requested. The helper supports local hit and miss features.


In [ ]:


def neighbors(cell: tuple[int, int], diagonal: bool = False) -> list[tuple[int, int]]:
    row, col = cell
    result = []
    for dr in (-1, 0, 1):
        for dc in (-1, 0, 1):
            if dr == dc == 0 or (not diagonal and abs(dr) + abs(dc) != 1):
                continue
            rr, cc = row + dr, col + dc
            if 0 <= rr < 10 and 0 <= cc < 10:
                result.append((rr, cc))
    return result


## `feature_vector`

Builds the MLP input for one candidate cell from only point-in-time observable information: previous shots and results, known sunk cells, and remaining ship lengths. It returns a fixed-length NumPy vector; hidden ship coordinates are not inputs.


In [ ]:


def feature_vector(cell, shots, outcomes, remaining_lengths, known_sunk_cells) -> np.ndarray:
    """Create features using only shots/results, sunk status and fleet lengths."""
    row, col = cell
    shot_map = dict(outcomes)
    sunk_cells = set(known_sunk_cells)
    unsunk_hits = {c for c, result in shot_map.items() if result == "hit" and c not in sunk_cells}
    misses = {c for c, result in shot_map.items() if result == "miss"}
    hit_distances = [abs(row - r) + abs(col - c) for r, c in unsunk_hits]
    center = abs(row - 4.5) + abs(col - 4.5)
    edge_distance = min(row, col, 9 - row, 9 - col)
    features = [
        row / 9, col / 9, center / 9, edge_distance / 4.5,
        float(row in (0, 9) and col in (0, 9)), row % 2, col % 2,
        len(shots) / 100, sum(v == "hit" for v in outcomes.values()) / 17,
        len(misses) / 100,
        len(set(neighbors(cell)) & unsunk_hits) / 4,
        len(set(neighbors(cell)) & misses) / 4,
        len(set(neighbors(cell, diagonal=True)) & unsunk_hits) / 4,
        sum(r == row for r, _ in unsunk_hits) / 5,
        sum(c == col for _, c in unsunk_hits) / 5,
        min(hit_distances, default=10) / 10,
        sum(remaining_lengths) / 17, len(remaining_lengths) / 5,
    ]
    return np.asarray(features, dtype=np.float32)


## `make_examples`

Simulates complete games and records snapshots at selected shot counts. For each unshot cell it builds observable features and uses the hidden layout only to assign the training label. Trajectories mix Strategist-like PDM, Commander-like PDM, noisy PDM, and random legal choices.


In [ ]:


def make_examples(games: int, seed: int, styles=("random", "edge", "center")):
    """Build point-in-time examples from gameplay-policy trajectories.

    Trajectory mix is fixed before evaluation: 40% Strategist PDM (random from
    the top 5%), 20% Commander PDM (first exact maximum), 25% noisy PDM (sample
    among its top quintile), and 15% random legal. Every snapshot remains
    grouped by complete layout/game for splitting.
    """
    rng = np.random.default_rng(seed)
    features, labels, game_ids = [], [], []
    for game_id in range(games):
        ships = generate_fleet(rng, styles[game_id % len(styles)])
        # Simulator-only truth for resolving shot outcomes. Never pass owner or ships to scoring.
        owner = {cell: i for i, ship in enumerate(ships) for cell in ship}
        shots: set[tuple[int, int]] = set()
        outcomes: dict[tuple[int, int], str] = {}
        sunk: set[int] = set()
        snapshot_set = set(SNAPSHOTS)
        for step in range(100):
            if step in snapshot_set:
                # Hidden cells are used only as binary training labels; features below use observable state.
                remaining = {c for i, ship in enumerate(ships) if i not in sunk for c in ship}
                for idx in range(100):
                    cell = (idx // 10, idx % 10)
                    if cell in shots:
                        continue
                    sunk_cells = {cell for cell, result in outcomes.items() if result == "sunk"}
                    remaining_lengths = [len(ship) for i, ship in enumerate(ships) if i not in sunk]
                    features.append(feature_vector(cell, shots, outcomes, remaining_lengths, sunk_cells))
                    labels.append(float(cell in remaining))
                    game_ids.append(game_id)
            if len(sunk) == len(ships):
                break
            available = [(r, c) for r in range(10) for c in range(10) if (r, c) not in shots]
            trajectory = int(rng.choice(100))
            remaining_lengths = [len(ship) for i, ship in enumerate(ships) if i not in sunk]
            if trajectory < 40:
                cell = choose_policy_cell("pdm", None, None, None, remaining_lengths, shots, outcomes, rng)
            elif trajectory < 60:
                cell = choose_policy_cell("commander", None, None, None, remaining_lengths, shots, outcomes, rng)
            elif trajectory < 85:
                candidates = density_candidates([len(ship) for i, ship in enumerate(ships) if i not in sunk], shots, outcomes)
                candidates.sort(key=lambda item: item["score"], reverse=True)
                cell = candidates[int(rng.integers(max(1, (len(candidates) + 4) // 5)))] ["cell"]
            else:
                cell = available[int(rng.integers(len(available)))]
            shots.add(cell)
            if cell in owner:
                outcomes[cell] = "hit"
                ship_id = owner[cell]
                if all(c in shots for c in ships[ship_id]):
                    sunk.add(ship_id)
                    for sunk_cell in ships[ship_id]:
                        if sunk_cell in shots:
                            outcomes[sunk_cell] = "sunk"
            else:
                outcomes[cell] = "miss"
    return (np.asarray(features, dtype=np.float32), np.asarray(labels, dtype=np.float32),
            np.asarray(game_ids, dtype=np.int32))


## `sigmoid`

Maps model logits to values between zero and one. In this experiment these are classifier outputs used for ranking and Brier-score evaluation; calibration is assessed rather than assumed.


In [ ]:


def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -30, 30)))


## `OccupancyMLP`

Implements a small two-layer neural network using NumPy arrays. `predict` runs the forward pass; `fit` performs mini-batch gradient updates with a fixed seed so training can be reproduced.


In [ ]:


class OccupancyMLP:
    """Small two-layer binary classifier implemented with NumPy only."""
    def __init__(self, input_size, seed=71, hidden=32):
        rng = np.random.default_rng(seed)
        self.w1 = rng.normal(0, np.sqrt(2 / input_size), (input_size, hidden)).astype(np.float32)
        self.b1 = np.zeros(hidden, dtype=np.float32)
        self.w2 = rng.normal(0, np.sqrt(2 / hidden), (hidden, 1)).astype(np.float32)
        self.b2 = np.zeros(1, dtype=np.float32)

    def predict(self, x):
        return sigmoid(np.tanh(x @ self.w1 + self.b1) @ self.w2 + self.b2).ravel()

    def fit(self, x, y, epochs=70, batch_size=512, learning_rate=0.015, seed=91):
        rng = np.random.default_rng(seed)
        for _ in range(epochs):
            for ids in np.array_split(rng.permutation(len(y)), max(1, (len(y) + batch_size - 1) // batch_size)):
                xb, yb = x[ids], y[ids, None]
                z1 = xb @ self.w1 + self.b1
                h = np.tanh(z1)
                p = sigmoid(h @ self.w2 + self.b2)
                dz2 = (p - yb) / len(ids)
                gw2, gb2 = h.T @ dz2, dz2.sum(axis=0)
                dz1 = (dz2 @ self.w2.T) * (1 - h * h)
                gw1, gb1 = xb.T @ dz1, dz1.sum(axis=0)
                self.w1 -= learning_rate * gw1
                self.b1 -= learning_rate * gb1
                self.w2 -= learning_rate * gw2
                self.b2 -= learning_rate * gb2


## `average_precision`

Computes average precision from ranked binary labels. This metric is more informative than raw accuracy for sparse ship-cell labels.


In [ ]:


def average_precision(y_true, scores):
    order = np.argsort(-scores, kind="stable")
    positives = float(np.sum(y_true))
    if positives == 0:
        return float("nan")
    ranked = y_true[order]
    precision = np.cumsum(ranked) / np.arange(1, len(ranked) + 1)
    return float(np.sum(precision * ranked) / positives)


## `score_report`

Collects average precision and Brier score for a set of predictions. Average precision measures ranking; Brier score measures squared probability error.


In [ ]:


def score_report(y, scores):
    return {"average_precision": average_precision(y, scores),
            "brier_score": float(np.mean((y - scores) ** 2))}


## `grouped_game_split`

Splits examples by complete game IDs, so snapshots from one fleet/game cannot appear across train, validation, and test partitions. This avoids leakage between highly related snapshots.


In [ ]:


def grouped_game_split(game_ids, games, seed, train_fraction=.70, validation_fraction=.15):
    """Split whole games/layouts so snapshots from one fleet cannot cross partitions."""
    unique_games = np.arange(games)
    np.random.default_rng(seed).shuffle(unique_games)
    n_train = int(games * train_fraction)
    n_val = int(games * validation_fraction)
    train_ids = set(unique_games[:n_train].tolist())
    validation_ids = set(unique_games[n_train:n_train + n_val].tolist())
    test_ids = set(unique_games[n_train + n_val:].tolist())
    ids = np.asarray(game_ids)
    return (np.isin(ids, list(train_ids)), np.isin(ids, list(validation_ids)),
            np.isin(ids, list(test_ids)), train_ids, validation_ids, test_ids)


## `density_candidates`

Implements the rule-based probability-density candidate scorer used to create PDM-style gameplay trajectories. It enumerates legal remaining-ship placements consistent with observed misses and hits, then ranks unshot cells.


In [ ]:


def density_candidates(remaining_lengths, shots, outcomes, sonar_observations=()):
    misses = {cell for cell, result in outcomes.items() if result == "miss"}
    sunk = {cell for cell, result in outcomes.items() if result == "sunk"}
    hits = {cell for cell, result in outcomes.items() if result == "hit"}
    heat = np.zeros((BOARD_SIZE, BOARD_SIZE), dtype=np.float64)
    target_mode = bool(hits)

    for length in remaining_lengths:
        for horizontal in (True, False):
            row_starts = range(BOARD_SIZE) if horizontal else range(BOARD_SIZE - length + 1)
            col_starts = range(BOARD_SIZE - length + 1) if horizontal else range(BOARD_SIZE)
            for row in row_starts:
                for col in col_starts:
                    cells = ([(row, col + offset) for offset in range(length)] if horizontal
                             else [(row + offset, col) for offset in range(length)])
                    keys = set(cells)
                    if keys & (misses | sunk):
                        continue
                    hit_count = sum(key in hits for key in keys)
                    weight = (0.05 if hit_count == 0 else 15 ** hit_count) if target_mode else 1.0
                    for r, c in cells:
                        if (r, c) not in shots:
                            heat[r, c] += weight

    if not target_mode and remaining_lengths:
        minimum_length = min(remaining_lengths)
        if minimum_length >= 2:
            for row in range(BOARD_SIZE):
                for col in range(BOARD_SIZE):
                    if (row + col) % minimum_length != 0:
                        heat[row, col] *= 0.15

    # Match PDMSolver's aggregate sonar constraint. `outcomes` insertion order
    # is shot order; only hits after the scan reduce its still-unshot count.
    ordered_shots = list(outcomes.items())
    for scan in sonar_observations:
        row, col = scan["center"]
        remaining_count = int(scan["count"])
        for (shot_row, shot_col), result in ordered_shots[int(scan.get("shotsObserved", 0)):]:
            if abs(shot_row - row) <= 1 and abs(shot_col - col) <= 1 and result in ("hit", "sunk"):
                remaining_count -= 1
        region = [(r, c) for r in range(row - 1, row + 2) for c in range(col - 1, col + 2)
                  if (r, c) not in shots]
        remaining_count = max(0, min(len(region), remaining_count))
        total_heat = float(np.sum(heat))
        expected = (sum(heat[r, c] for r, c in region) / (total_heat or 1) * sum(remaining_lengths))
        if remaining_count == 0:
            for r, c in region:
                heat[r, c] = 0
        else:
            scale = max(.25, min(4, remaining_count / max(expected, .25)))
            for r, c in region:
                heat[r, c] *= scale

    result = []
    for row in range(BOARD_SIZE):
        for col in range(BOARD_SIZE):
            if (row, col) not in shots:
                result.append({"cell": (row, col), "score": float(heat[row, col])})
    return result


## `choose_policy_cell`

Selects a legal shot for one of the generator policies. This controls which game states appear in training data; it is not the learned occupancy model.


In [ ]:


def choose_policy_cell(policy, model, mean, scale, remaining_lengths, shots, outcomes, rng, model_weight=0.2):
    if policy == "random":
        legal = [(r, c) for r in range(BOARD_SIZE) for c in range(BOARD_SIZE) if (r, c) not in shots]
        return legal[int(rng.integers(len(legal)))]
    candidates = density_candidates(remaining_lengths, shots, outcomes)
    if policy == "commander":
        return max(candidates, key=lambda item: item["score"])["cell"]
    if policy == "hybrid" and model_weight > 0:
        sunk_cells = {cell for cell, result in outcomes.items() if result == "sunk"}
        features = np.stack([
            feature_vector(item["cell"], shots, outcomes, remaining_lengths, sunk_cells)
            for item in candidates
        ])
        model_scores = model.predict(np.clip((features - mean) / scale, -5, 5).astype(np.float32))
        rule_scores = np.asarray([max(0, item["score"]) for item in candidates], dtype=np.float64)
        rule_total = rule_scores.sum() or 1
        model_total = model_scores.sum() or 1
        combined = (1 - model_weight) * (rule_scores / rule_total) + model_weight * (model_scores / model_total)
    else:
        combined = np.asarray([item["score"] for item in candidates], dtype=np.float64)
    maximum = np.max(combined)
    threshold = maximum * 0.95 if policy == "pdm" else maximum
    best = np.flatnonzero(combined >= threshold)
    return candidates[int(rng.choice(best))]["cell"]


## `evaluate_full_games`

Runs complete held-out games for baseline and hybrid policies and aggregates gameplay metrics such as shots to sink, hit rate, and invalid actions. This checks whether better cell ranking transfers to gameplay.


In [ ]:


def evaluate_full_games(model, mean, scale, games=150, seed=20266028, model_weight=0.2, layouts=None):
    if layouts is None:
        rng = np.random.default_rng(seed)
        layouts = [generate_fleet(rng) for _ in range(games)]
    games = len(layouts)
    results = {}
    for policy in ("random", "pdm", "hybrid"):
        shot_counts, hit_counts, invalid_counts, decision_times = [], [], [], []
        for game_index, ships in enumerate(layouts):
            local_rng = np.random.default_rng(seed + game_index * 3)
            owner = {cell: i for i, ship in enumerate(ships) for cell in ship}
            shots: set[tuple[int, int]] = set()
            outcomes: dict[tuple[int, int], str] = {}
            sunk_ids: set[int] = set()
            invalid = 0
            while len(sunk_ids) < len(ships):
                start = perf_counter()
                remaining_lengths = [len(ship) for i, ship in enumerate(ships) if i not in sunk_ids]
                cell = choose_policy_cell(policy, model, mean, scale, remaining_lengths, shots, outcomes, local_rng,
                                          model_weight=model_weight)
                decision_times.append(perf_counter() - start)
                if cell in shots or not (0 <= cell[0] < 10 and 0 <= cell[1] < 10):
                    invalid += 1
                    break
                shots.add(cell)
                if cell in owner:
                    outcomes[cell] = "hit"
                    ship_id = owner[cell]
                    if all(part in shots for part in ships[ship_id]):
                        sunk_ids.add(ship_id)
                        for part in ships[ship_id]:
                            if part in shots:
                                outcomes[part] = "sunk"
                else:
                    outcomes[cell] = "miss"
            shot_counts.append(len(shots))
            hit_counts.append(sum(value in ("hit", "sunk") for value in outcomes.values()))
            invalid_counts.append(invalid)
        total_shots = sum(shot_counts)
        results[policy] = {
            "games": games,
            "mean_shots_to_sink": float(np.mean(shot_counts)),
            "median_shots_to_sink": float(np.median(shot_counts)),
            "mean_hit_rate": float(sum(hit_counts) / total_shots) if total_shots else 0,
            "invalid_or_repeated_shots": int(sum(invalid_counts)),
            "mean_decision_time_ms": float(np.mean(decision_times) * 1000),
            "shots_per_layout": shot_counts,
        }
    paired = np.asarray(results["hybrid"]["shots_per_layout"]) - np.asarray(results["pdm"]["shots_per_layout"])
    bootstrap_rng = np.random.default_rng(seed + 99000)
    bootstrap_means = np.mean(paired[bootstrap_rng.integers(0, len(paired), (2000, len(paired)))], axis=1)
    results["paired_hybrid_minus_pdm_shots"] = {
        "mean": float(np.mean(paired)),
        "bootstrap_95pct_ci": [float(np.quantile(bootstrap_means, .025)), float(np.quantile(bootstrap_means, .975))],
        "layout_count": int(len(paired)),
    }
    for policy in ("random", "pdm", "hybrid"):
        results[policy].pop("shots_per_layout")
    results["hybrid_minus_pdm_mean_shots"] = (
        results["hybrid"]["mean_shots_to_sink"] - results["pdm"]["mean_shots_to_sink"]
    )
    return results


## `run`

Coordinates data generation, game-grouped splitting, model training, validation-only blend-weight selection, held-out evaluation, and export of the model/report artifacts. The final test split is used for reporting, not weight tuning.


In [ ]:


def run(output: Path, games: int, seed: int, export_frontend=True):
    output.mkdir(parents=True, exist_ok=True)
    x, y, game_ids = make_examples(games, seed)
    train, val, test, train_games, val_games, test_games = grouped_game_split(game_ids, games, seed + 1)

    # Standardization parameters come from training games only.
    mean, scale = x[train].mean(axis=0), x[train].std(axis=0)
    scale[scale < 1e-6] = 1
    z = np.clip((x - mean) / scale, -5, 5).astype(np.float32)
    model = OccupancyMLP(x.shape[1], seed=seed + 2)
    model.fit(z[train], y[train])
    export = {"format": "adaptive-battleship-occupancy-mlp-v1", "features": list(FEATURE_NAMES),
              "training_seed": seed, "training_games": games,
              "trajectory_policy_mix": {"strategist_pdm_top_5pct": 0.40, "commander_pdm_exact_max": 0.20,
                                        "noisy_pdm_top_quintile": 0.25, "random_legal": 0.15},
              "validation_candidate_weights": [0.0, 0.05, 0.1, 0.2],
              "hidden_activation": "tanh", "output_activation": "sigmoid",
              "weights": {k: v.tolist() for k, v in {
                  "w1": model.w1, "b1": model.b1, "w2": model.w2, "b2": model.b2,
                  "mean": mean, "scale": scale}.items()}}
    validation_rng = np.random.default_rng(seed + 4000)
    validation_layouts = [generate_fleet(validation_rng, style)
                          for style in ("random", "edge", "center") for _ in range(30)]
    validation_weights = {}
    validation_baselines = None
    for weight in (0.0, 0.05, 0.1, 0.2):
        validation_results = evaluate_full_games(model, mean, scale, seed=seed + 4000,
                                                 model_weight=weight, layouts=validation_layouts)
        validation_weights[str(weight)] = validation_results["hybrid"]["mean_shots_to_sink"]
        if weight == 0.0:
            validation_baselines = {
                "pdm_only_mean_shots_to_sink": validation_results["pdm"]["mean_shots_to_sink"],
                "random_legal_mean_shots_to_sink": validation_results["random"]["mean_shots_to_sink"],
            }
    selected_weight = min((float(weight) for weight in validation_weights),
                          key=lambda weight: validation_weights[str(weight)])
    export["recommended_hybrid_weight"] = selected_weight
    test_rng = np.random.default_rng(seed + 5000)
    test_layouts_by_style = {style: [generate_fleet(test_rng, style) for _ in range(60)]
                             for style in ("random", "edge", "center", "corner")}
    policy_evaluation = {style: evaluate_full_games(model, mean, scale, seed=seed + 5000,
                            model_weight=selected_weight, layouts=layouts)
                         for style, layouts in test_layouts_by_style.items()}
    export["recommended_hybrid_weight"] = selected_weight
    model_score = model.predict(z[test])
    # Observable base rate: known remaining ship segments / unshot cells.
    # Since snapshots are bundled by game, calculate it from the visible feature.
    test_shots_fraction = x[test, FEATURE_NAMES.index("shots_fraction")]
    test_remaining_fraction = x[test, FEATURE_NAMES.index("remaining_segments_fraction")]
    baseline_score = np.clip(test_remaining_fraction * 17 / np.maximum(100 * (1 - test_shots_fraction), 1), 0, 1)
    heldout_x, heldout_y, _ = make_examples(60, seed + 6000, styles=("corner",))
    heldout_z = np.clip((heldout_x - mean) / scale, -5, 5).astype(np.float32)
    heldout_shots = heldout_x[:, FEATURE_NAMES.index("shots_fraction")]
    heldout_remaining = heldout_x[:, FEATURE_NAMES.index("remaining_segments_fraction")]
    heldout_base_rate = np.clip(heldout_remaining * 17 / np.maximum(100 * (1 - heldout_shots), 1), 0, 1)
    report = {
        "experiment": "ship_cell_occupancy",
        "seed": seed,
        "games": games,
        "trajectory_policy_mix": {"strategist_pdm_top_5pct": 0.40, "commander_pdm_exact_max": 0.20,
                                  "noisy_pdm_top_quintile": 0.25, "random_legal": 0.15},
        "training_fleet_styles": ["random", "edge", "center"],
        "held_out_fleet_style": "corner",
        "split_by_complete_game": {"train": len(train_games), "validation": len(val_games), "test": len(test_games)},
        "rows": {"train": int(train.sum()), "validation": int(val.sum()), "test": int(test.sum())},
        "features": list(FEATURE_NAMES),
        "held_out_test": {
            "positive_rate": float(y[test].mean()),
            "observable_base_rate": score_report(y[test], baseline_score),
            "numpy_mlp": score_report(y[test], model_score),
        },
        "held_out_placement_style": {
            "style": "corner", "layout_count": 60,
            "positive_rate": float(heldout_y.mean()),
            "observable_base_rate": score_report(heldout_y, heldout_base_rate),
            "numpy_mlp": score_report(heldout_y, model.predict(heldout_z)),
        },
        "validation_policy_weight_selection": {
            "layout_count": len(validation_layouts),
            "candidate_weights_predeclared": [0.0, 0.05, 0.1, 0.2],
            "baselines": validation_baselines,
            "mean_shots_by_model_weight": validation_weights,
            "selected_model_weight": selected_weight,
        },
        "full_game_policy_evaluation_on_independent_layouts": policy_evaluation,
        "scan_usage": "not_applicable; occupancy-only targeting experiment has no scan action selection",
        "limitations": [
            "Synthetic random, edge-biased, and center-biased training fleets; corner-biased layout style is held out from training.",
            "Full-game evaluation is a one-sided shots-to-sink comparison, not a measured two-sided win rate.",
            "The browser uses a direct JavaScript inference port and a separate local fleet-history prior; this report evaluates only the offline synthetic-model hybrid.",
        ],
    }
    np.savez_compressed(output / "occupancy_mlp.npz", w1=model.w1, b1=model.b1,
                        w2=model.w2, b2=model.b2, mean=mean, scale=scale)
    (output / "occupancy_mlp.json").write_text(json.dumps(export, separators=(",", ":")), encoding="utf-8")
    js_export = "// Generated by model/src/occupancy_experiment.py; do not edit by hand.\n"
    js_export += "export const OCCUPANCY_MLP = " + json.dumps(export, separators=(",", ":")) + ";\n"
    (output / "occupancy_mlp.js").write_text(js_export, encoding="utf-8")
    frontend_model = Path(__file__).resolve().parents[2] / "frontend" / "src" / "ai" / "occupancy-model.generated.js"
    frontend_model.parent.mkdir(parents=True, exist_ok=True)
    if export_frontend:
        frontend_model.write_text(js_export, encoding="utf-8")
    (output / "results.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report


## `run_replicates`

Repeats the experiment across independent random seeds and summarizes variation. Independent runs help show whether results depend on a lucky seed.


In [ ]:


def run_replicates(output: Path, games: int, seeds: list[int]):
    """Run independent generator/training seeds; first seed exports the browser model."""
    reports = []
    for index, seed in enumerate(seeds):
        run_dir = output if index == 0 else output / "replicates" / str(seed)
        reports.append(run(run_dir, games, seed, export_frontend=(index == 0)))
    summary = {
        "experiment": "ship_cell_occupancy_multi_seed", "seeds": seeds,
        "exported_model_seed": seeds[0], "replicates": reports,
        "held_out_style_policy_means": {
            style: {policy: float(np.mean([
                rep["full_game_policy_evaluation_on_independent_layouts"][style][policy]["mean_shots_to_sink"]
                for rep in reports])) for policy in ("random", "pdm", "hybrid")}
            for style in ("random", "edge", "center", "corner")
        },
    }
    style_means = {style: float(np.mean([
        rep["full_game_policy_evaluation_on_independent_layouts"][style]["paired_hybrid_minus_pdm_shots"]["mean"]
        for rep in reports])) for style in ("random", "edge", "center", "corner")}
    per_seed_pass = all(all(
        rep["full_game_policy_evaluation_on_independent_layouts"][style]["paired_hybrid_minus_pdm_shots"]["bootstrap_95pct_ci"][1] < 0
        for style in ("random", "edge", "center", "corner")) for rep in reports)
    enable_weight = reports[0]["validation_policy_weight_selection"]["selected_model_weight"]
    enable_weight = enable_weight if per_seed_pass and all(value < 0 for value in style_means.values()) else 0.0
    summary["runtime_weight_decision"] = {
        "enabled_weight": enable_weight,
        "gate": "enable only if every independent seed and every held-out placement style has a paired 95% bootstrap upper bound below zero, with aggregate improvement in every style",
        "mean_paired_hybrid_minus_pdm_shots_by_style": style_means,
        "reason": "Held-out gameplay evidence did not pass the predeclared consistency gate." if enable_weight == 0 else "Held-out gameplay evidence passed the predeclared consistency gate.",
    }
    # The first replicate is the exported model. The held-out set is used only
    # as an enable/disable evidence gate, never to tune the selected blend weight.
    first_export_path = output / "occupancy_mlp.json"
    first_export = json.loads(first_export_path.read_text(encoding="utf-8"))
    first_export["validation_selected_hybrid_weight"] = first_export.get("recommended_hybrid_weight", 0.0)
    first_export["recommended_hybrid_weight"] = enable_weight
    first_export["gameplay_gate_passed"] = enable_weight > 0
    js_export = "// Generated by model/src/occupancy_experiment.py; do not edit by hand.\n"
    js_export += "export const OCCUPANCY_MLP = " + json.dumps(first_export, separators=(",", ":")) + ";\n"
    first_export_path.write_text(json.dumps(first_export, separators=(",", ":")), encoding="utf-8")
    (output / "occupancy_mlp.js").write_text(js_export, encoding="utf-8")
    frontend_model = Path(__file__).resolve().parents[2] / "frontend" / "src" / "ai" / "occupancy-model.generated.js"
    frontend_model.write_text(js_export, encoding="utf-8")
    (output / "results.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary


## `<entry>`

Command-line entry point. It parses command-line options and starts the offline experiment. This copied cell is included to explain how the original script is launched; do not run it as a notebook cell or treat this notebook as the application entry point.


In [ ]:


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--games", type=int, default=600)
    parser.add_argument("--seed", type=int, default=20261028)
    parser.add_argument("--seeds", type=str, default="20261028,20261029,20261030",
                        help="independent seeds; the first seed exports the browser artifact")
    parser.add_argument("--output", type=Path, default=Path(__file__).resolve().parents[1] / "artifacts")
    args = parser.parse_args()
    seeds = [int(value.strip()) for value in args.seeds.split(",") if value.strip()]
    if args.seed != 20261028 and seeds == [20261028, 20261029, 20261030]:
        seeds[0] = args.seed
    print(json.dumps(run_replicates(args.output, args.games, seeds), indent=2))
